# CHAI — keyword-centered ego-network pipeline

**Stage 0** (provenance + sampling) and **Stage 1** (ego-network extraction) for the
keyword-centered AMR analysis of the *fairness*-MapAIE corpus.

The pipeline is keyword-agnostic: everything keys on the global variable `KEY_WORD`,
so switching from `fairness` to `transparency` later requires changing one line.

**What this notebook does**
- *Stage 0.1* clean the `fairness_MapAIE.csv` pool (drop references, run-ons, fragments).
- *Stage 0.2* draw a 100-sentence working sample (uniform, with a soft per-document cap).
- *Stage 0.3* attach an `actor_type` to each sampled occurrence via `doc_id -> title -> Sector`.
- *Stage 1.1* parse the sampled sentences to AMR (`amrlib`).
- *Stage 1.2* extract the typed ego-network (parents / children / siblings, radius 1-2-3)
  of the keyword node in each graph.

**Two manual interludes** are flagged where human judgment is required:
verifying the `actor_type` map, and correcting the `fairness`/`fair-01` parser artifact
in metAMoRphosED before extraction.

> `doc_id` is **not** a positional index: in the generating notebook the id is the txt
> filename itself (`doc_id = int(fname.replace(".txt", ""))`), so the sentence -> document
> link is exact. Only `doc_id -> actor_type` requires alignment work.

## Configuration

In [ ]:
import os, re, json, difflib
import numpy as np
import pandas as pd

# ---- global, keyword-agnostic configuration -------------------------------
KEY_WORD      = "fairness"               # change this single line to retarget the pipeline
KEY_CONCEPTS  = {"fairness", "fair-01"}  # AMR concept labels that realize KEY_WORD
                                         # (kept explicit because of the fairness/fair-01 artifact)

N_SAMPLE      = 100
MAX_PER_DOC   = 10          # soft cap: prevents one prolix doc from dominating the sample
RANDOM_SEED   = 42
MIN_TOK, MAX_TOK = 4, 80    # discard fragments and run-on segmentation failures
RADII         = (1, 2, 3)

# ---- paths (adjust to your local clone) -----------------------------------
CSV_PATH   = "fairness_MapAIE.csv"
XLSX_PATH  = "AI_Ethics_Manifestos.xlsx"
XLSX_SHEET = "List of documents"
TXT_DIR    = "./mapaie/txt"             # local clone of gitlab.telecom-paris.fr/tiphaine.viard/mapaie
AMR_OUT    = f"{KEY_WORD}_sample_penmans.txt"
MAP_OUT    = f"{KEY_WORD}_sample_doc_actor_map.csv"

RUN_PARSER = False   # set True locally once amrlib + a model are installed

rng = np.random.default_rng(RANDOM_SEED)

## Stage 0.1 — Load and clean the keyword pool

The raw CSV mixes genuine usages with bibliographic references, URLs, and run-on
segments where the keyword is not a usable node. The filters below are conservative
and reported, not silent; tune `MIN_TOK`/`MAX_TOK` and the citation pattern as needed.

In [ ]:
raw = pd.read_csv(CSV_PATH)
raw["sentence"] = raw["sentence"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
raw["ntok"]     = raw["sentence"].str.findall(r"\b\w+\b").apply(len)

has_kw = raw["sentence"].str.contains(rf"\b{KEY_WORD}\b", case=False, regex=True)
url    = raw["sentence"].str.contains(r"https?:/|\.pdf|ssrn|doi", case=False, regex=True)
cite   = raw["sentence"].str.contains(r"Proceedings of|\bpp\.\s*\d|\bISBN\b|\bvol\.\s*\d", regex=True)
keep   = has_kw & raw["ntok"].between(MIN_TOK, MAX_TOK) & ~url & ~cite

clean = raw[keep].reset_index(drop=True)
print(f"raw {len(raw)} -> clean {len(clean)} (dropped {len(raw)-len(clean)})")
print(f"  no keyword : {int((~has_kw).sum())}")
print(f"  too short  : {int((raw['ntok']<MIN_TOK).sum())}")
print(f"  too long   : {int((raw['ntok']>MAX_TOK).sum())}")
print(f"  url/doi     : {int(url.sum())}")
print(f"  citation    : {int(cite.sum())}")
print(f"clean pool spans {clean['doc_id'].nunique()} distinct documents")

## Stage 0.2 — Draw the working sample

A uniform draw already *reflects* the natural distribution (a document with many
keyword sentences contributes proportionally more), which is the desired behaviour.
The per-document cap is only a safety rail against degenerate concentration, not a
forced balancing. Sampling at sentence level; note a sentence may yield more than one
keyword node downstream.

In [ ]:
pool = clean.sample(frac=1, random_state=RANDOM_SEED)   # shuffle
picked, per_doc = [], {}
for _, row in pool.iterrows():
    d = row["doc_id"]
    if per_doc.get(d, 0) >= MAX_PER_DOC:
        continue
    picked.append(row)
    per_doc[d] = per_doc.get(d, 0) + 1
    if len(picked) >= N_SAMPLE:
        break

sample = pd.DataFrame(picked).reset_index(drop=True)
sample.insert(0, "sample_id", [f"s{i:03d}" for i in range(len(sample))])
print(f"sampled {len(sample)} sentences from {sample['doc_id'].nunique()} distinct documents")
print("per-document counts:", sorted(per_doc.values(), reverse=True))
sample.head()

## Stage 0.3 — Attach `actor_type` (`doc_id -> title -> Sector`)

The metadata lives in the Excel `List of documents` sheet (no numeric id column), so the
join is by content: read a title candidate from each `{doc_id}.txt`, fuzzy-match it to
`Name of the document`, then read `Sector`. Only the documents present in the 100-sentence
sample need resolving, so manual verification stays light.

If the mapaie repo's `all_manifestos.csv` turns out to carry an explicit id/filename column,
prefer an exact join over this fuzzy step.

In [ ]:
meta = pd.read_excel(XLSX_PATH, sheet_name=XLSX_SHEET)
meta = meta.rename(columns={"Name of the document": "title", "Sector": "sector"})
meta = meta.dropna(subset=["title"]).copy()
meta["title_norm"] = meta["title"].astype(str).str.lower().str.strip()
titles = meta["title_norm"].tolist()

def normalize_sector(s):
    """Collapse compound/hierarchical Sector labels to a base category.
    Returns (base_category, is_compound). Compound rows are flagged, not forced:
    decide their final label by hand (keeps the six-category typology auditable)."""
    if not isinstance(s, str):
        return (None, False)
    compound = ("," in s)
    base = s.split(">")[0].split(",")[0].strip().lower()
    return (base or None, compound)

print(f"metadata rows with a title: {len(meta)}")
print("raw sectors present:", sorted(meta["sector"].dropna().unique().tolist())[:8], "...")

In [ ]:
def read_title_candidate(doc_id, txt_dir=TXT_DIR, n_lines=15):
    """Heuristic title: first reasonably long non-numeric line of the txt file."""
    path = os.path.join(txt_dir, f"{doc_id}.txt")
    if not os.path.exists(path):
        return None
    with open(path, encoding="utf-8", errors="replace") as f:
        for line in (next(f, "") for _ in range(n_lines)):
            s = line.strip()
            if len(s) >= 12 and re.search(r"[A-Za-z]", s) and not s.isdigit():
                return s
    return None

def resolve_actor(doc_id):
    """Return (matched_title, base_sector, is_compound, score) for a doc_id."""
    cand = read_title_candidate(doc_id)
    if cand is None:
        return (None, None, False, 0.0)
    hit = difflib.get_close_matches(cand.lower(), titles, n=1, cutoff=0.0)
    if not hit:
        return (None, None, False, 0.0)
    score = difflib.SequenceMatcher(None, cand.lower(), hit[0]).ratio()
    row = meta.loc[meta["title_norm"] == hit[0]].iloc[0]
    base, compound = normalize_sector(row["sector"])
    return (row["title"], base, compound, round(score, 3))

if os.path.isdir(TXT_DIR):
    recs = []
    for d in sorted(sample["doc_id"].unique()):
        t, sec, comp, sc = resolve_actor(d)
        recs.append({"doc_id": d, "matched_title": t, "actor_type": sec, "compound_sector": comp,
                     "match_score": sc, "needs_review": (sc < 0.60) or (sec is None) or comp})
    doc_map = pd.DataFrame(recs)
    doc_map.to_csv(MAP_OUT, index=False)
    print(f"resolved {len(doc_map)} documents -> {MAP_OUT}")
    print(f"flagged for manual review: {int(doc_map['needs_review'].sum())} "
          f"(low score, unmatched, or compound sector)")
else:
    print(f"[skipped] {TXT_DIR} not found — run locally with the mapaie clone present.")
    doc_map = pd.DataFrame(columns=["doc_id","matched_title","actor_type","compound_sector","match_score","needs_review"])

### Manual interlude 1 — verify the actor map

Open `*_doc_actor_map.csv`, check the rows where `needs_review` is True, and correct
`actor_type` by hand. Record corrections in `MANUAL_OVERRIDES` so the notebook stays
reproducible, then re-run the merge below.

In [ ]:
MANUAL_OVERRIDES = {
    # doc_id: "actor_type",
    # e.g. 52: "national authority",
}
if len(doc_map):
    for d, sec in MANUAL_OVERRIDES.items():
        doc_map.loc[doc_map["doc_id"] == d, ["actor_type", "needs_review"]] = [sec, False]
    sample = sample.merge(doc_map[["doc_id", "actor_type"]], on="doc_id", how="left")
    print(sample["actor_type"].value_counts(dropna=False))
    print("\\nReminder (H2): if a type is too sparse for the intra-type contrast, "
          "add a small documented top-up rather than re-stratifying the whole sample.")
else:
    sample["actor_type"] = pd.NA
    print("actor_type left empty — resolve Stage 0.3 locally first.")

## Stage 1.1 — Parse the sample to AMR

Runs `amrlib` locally. Output is written in the project's Penman-block format
(`# ::id`, `# ::snt`, one blank line between graphs), so it loads back cleanly.

In [ ]:
if RUN_PARSER:
    import amrlib
    stog = amrlib.load_stog_model()                     # e.g. parse_xfm_bart_base
    graphs = stog.parse_sents(sample["sentence"].tolist())
    with open(AMR_OUT, "w", encoding="utf-8") as f:
        for sid, did, g in zip(sample["sample_id"], sample["doc_id"], graphs):
            f.write(f"# ::id {sid} ::doc_id {did}\n")
            body = g if g.lstrip().startswith("#") else g
            f.write(body.strip() + "\n\n")
    print(f"parsed {len(graphs)} sentences -> {AMR_OUT}")
else:
    print("[skipped] set RUN_PARSER = True locally to parse with amrlib.")

### Manual interlude 2 — correct the `fairness`/`fair-01` artifact

Before extraction, open `*_sample_penmans.txt` in metAMoRphosED and reconcile the
keyword node case by case: decide, per sentence, whether the occurrence is predicative
(`fair-01`, a predicate with arguments) or nominal (`fairness`, a property). Do **not**
mass-replace one with the other — the two carry different argument structures. Document
the decisions; this is part of the analysis, not preprocessing.

## Stage 1.2 — Typed ego-network extraction

The core, keyword-agnostic brick. For each graph it locates the keyword node(s)
(exact concept match against `KEY_CONCEPTS`, not substring), then returns:
its position, its typed neighbours (parents / children / siblings = co-arguments),
and the directed subgraph at each radius in `RADII`.

In [ ]:
import penman
import networkx as nx

def amr_to_nx(g: "penman.Graph"):
    """Penman graph -> directed networkx graph; returns (G, concept_of_var)."""
    G = nx.DiGraph()
    concept = {i.source: i.target for i in g.instances()}
    for v, c in concept.items():
        G.add_node(v, concept=c, kind="instance")
    for s, r, o in g.edges():            # relations between instances
        G.add_edge(s, o, role=r)
    for s, r, o in g.attributes():       # constants / leaves (e.g. :polarity -, names)
        leaf = f"{s}{r}__{o}"
        G.add_node(leaf, concept=str(o), kind="attr")
        G.add_edge(s, leaf, role=r)
    return G, concept

def find_keyword_vars(concept, key_concepts=KEY_CONCEPTS):
    return [v for v, c in concept.items() if c in key_concepts]

def typed_neighbours(G, var):
    parents  = [(p, G.nodes[p].get("concept", "?"), G[p][var]["role"]) for p in G.predecessors(var)]
    children = [(c, G.nodes[c].get("concept", "?"), G[var][c]["role"]) for c in G.successors(var)]
    siblings = []                        # co-arguments under a shared parent ("adelphes")
    for p in G.predecessors(var):
        for c in G.successors(p):
            if c != var:
                siblings.append((c, G.nodes[c].get("concept", "?"), G[p][c]["role"]))
    return parents, children, siblings

def ego_subgraph(G, var, radius):
    """Directed subgraph of the undirected radius-r ball around `var`."""
    ball = nx.ego_graph(G.to_undirected(as_view=True), var, radius=radius)
    return G.subgraph(ball.nodes).copy()

def position(G, var):
    if G.in_degree(var) == 0:  return "root"
    if G.out_degree(var) == 0: return "leaf"
    return "internal"

In [ ]:
def extract_all(amr_path=AMR_OUT):
    """Run the brick over every parsed graph; return a tidy DataFrame + per-node subgraphs."""
    graphs = penman.load(amr_path)       # list of penman.Graph, metadata preserved
    rows, subgraphs = [], {}
    for g in graphs:
        sid = g.metadata.get("id", g.top)
        did = g.metadata.get("doc_id")
        G, concept = amr_to_nx(g)
        for k, var in enumerate(find_keyword_vars(concept)):
            node_id = f"{sid}#{k}"
            par, ch, sib = typed_neighbours(G, var)
            subgraphs[node_id] = {r: ego_subgraph(G, var, r) for r in RADII}
            rows.append({
                "node_id": node_id, "sample_id": sid, "doc_id": did,
                "keyword_concept": concept[var], "position": position(G, var),
                "n_parents": len(par), "n_children": len(ch), "n_siblings": len(sib),
                "parent_rels":     [r for _, _, r in par],
                "parent_concepts": [c for _, c, _ in par],
                "child_rels":      [r for _, _, r in ch],
                "sibling_concepts":[c for _, c, _ in sib],
                **{f"r{r}_nodes": subgraphs[node_id][r].number_of_nodes() for r in RADII},
            })
    return pd.DataFrame(rows), subgraphs

if os.path.exists(AMR_OUT):
    ego_df, ego_subgraphs = extract_all()
    if "actor_type" in sample.columns:
        ego_df = ego_df.merge(sample[["sample_id", "actor_type"]], on="sample_id", how="left")
    print(f"extracted {len(ego_df)} keyword nodes from {ego_df['sample_id'].nunique()} sentences")
    display(ego_df.head())
    print("\\nposition distribution:\\n", ego_df["position"].value_counts())
else:
    print(f"[skipped] {AMR_OUT} not found — run Stage 1.1 first.")

## Output and next step

`ego_df` holds the per-occurrence structural summary; `ego_subgraphs[node_id][r]` holds
the directed ego-network at each radius, ready for **Stage 2** (pairwise similarity —
SMATCH held constant against the whole-graph baseline to isolate the zoom effect, then
SEMCAT/WL), which feeds the H1 (separability) and H2 (usage regime) tests.

The two manual interludes — actor-map verification and the `fairness`/`fair-01`
reconciliation — must be completed before the Stage 2 numbers are interpretable.

In [ ]:
# persist artefacts for Stage 2
sample.to_csv(f"{KEY_WORD}_sample_100.csv", index=False)
if "ego_df" in dir():
    ego_df.to_csv(f"{KEY_WORD}_ego_summary.csv", index=False)
    print("saved:", f"{KEY_WORD}_sample_100.csv", "+", f"{KEY_WORD}_ego_summary.csv")
else:
    print("saved:", f"{KEY_WORD}_sample_100.csv")